In [ ]:
import ast
import json
import random
import re
import textwrap
from utils import tool_parse

In [ ]:
def json_toolcall_to_python(tool_calls: dict, markdown_format=True) -> str:
    """Convert a JSON tool-call into a Python-style function call string."""

    def format_value(value):
        """Format Python values for code representation."""
        if isinstance(value, str):
            return f'"{value}"'
        elif isinstance(value, bool):
            return "True" if value else "False"
        elif value is None:
            return "None"
        elif isinstance(value, (int, float)):
            return str(value)
        elif isinstance(value, list):
            return "[" + ", ".join(format_value(v) for v in value) + "]"
        elif isinstance(value, dict):
            return (
                "{"
                + ", ".join(
                    f"{format_value(k)}: {format_value(v)}" for k, v in value.items()
                )
                + "}"
            )
        else:
            raise ValueError(f"Unsupported argument type: {type(value)}")

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)
    if isinstance(tool_calls, dict):
        tool_calls = [tool_calls]
    returns = []

    for tool_call in tool_calls:
        func_name = tool_call.get("name")
        arguments = tool_call.get("arguments", {})

        if not func_name:
            raise ValueError("Missing function name in tool call.")

        args_str = ", ".join(
            f"{key}={format_value(val)}" for key, val in arguments.items()
        )
        returns.append(f"{func_name}({args_str})")
    if markdown_format:
        return f"```python\n{'\n'.join(returns)}\n```"
    return returns


def json_tooldef_to_python(tools: list, indent=None) -> str:
    """
    Convert JSON tool descriptions into Python function definitions (as a string).

    - Maps JSON schema types to Python types
    - Handles required vs optional parameters
    - Converts enums to Literal types
    - Adds docstrings from descriptions
    """

    def map_type(prop: dict) -> str:
        """Map JSON schema types to Python type hints."""
        t = prop.get("type", "any")
        if not isinstance(t, str):
            t = 'any'

        # Handle enum -> Literal
        if "enum" in prop or "enums" in prop:
            values = prop.get("enum") or prop.get("enums")
            literals = ", ".join(repr(v) for v in values)
            return f"Literal[{literals}]"

        return {
            "string": "str",
            "integer": "int",
            "int": "int",
            "number": "float",
            "float": "float",
            "boolean": "bool",
            "array": "List[Any]",
            "list": "List[Any]",
            "object": "object",
            "any": "Any",
            "null": "None",
        }.get(t, t)

    lines = []
    if indent is None:
        indent = random.randint(0, 6)

    if isinstance(tools, str):
        try:
            tools = tool_parse(tools)
        except Exception as E:
            print(tools)
            raise E

    for tool in tools:
        if isinstance(tool, str):
            try:
                tool = tool_parse(tools)
            except Exception as E:
                print(tool)
                raise E
        # print(tool)
        name = tool["name"]
        desc = tool.get("description", "")
        params = tool.get("parameters", {})
        props = params.get("properties", {})
        required = set(params.get("required", []))

        args_list = []
        doc_params = []

        for p_name, p_info in props.items():
            py_type = map_type(p_info)
            has_default = "default" in p_info
            default_val = p_info.get("default")
            
            # Required vs optional
            if p_name in required:
                arg = f"{p_name}: {py_type}"
            else:
                if has_default:
                    arg = f"{p_name}: Optional[{py_type}] = {format_default(default_val)}"
                else:
                    arg = f"{p_name}: Optional[{py_type}] = None"
            args_list.append(arg)

            # Parameter doc
            p_desc = p_info.get("description")
            if p_desc:
                doc_params.append(f"{' '*indent}{p_name}: {p_desc}")

        args_str = ", ".join(args_list)
        
        # Build function string
        func_def = f"def {name}({args_str}):\n"
        func_def += f'{" "*indent}"""{desc}\n\n'
        if doc_params:
            func_def += f"{' '*indent}Args:\n" + "\n".join(doc_params) + "\n"
        func_def += f'{" "*indent}"""\n'
        func_def += f"{' '*(indent)}pass\n"

        lines.append(func_def)

    return "\n\n".join(lines)

In [ ]:
tools = [
    {
        "name": "retrieve_payment_status",
        "description": "Get payment status of a transaction",
        "parameters": {
            "type": "object",
            "properties": {
                "transaction_id": {
                    "type": "string",
                    "description": "The transaction id.",
                    "default": 'buy+123'
                }
            },
            "required": ["transaction_id"],
        },
    },
    {
        "name": "retrieve_payment_date",
        "description": "Get payment date of a transaction",
        "parameters": {
            "type": "object",
            "properties": {
                "transaction_id": {
                    "type": "string",
                    "description": "The transaction id.",
                },
                "additional_inputs": {"type": "string", "enums": ["YES", "NO"]},
            },
            "required": ["transaction_id"],
        },
    },
]

json_call = json.dumps(
    [
        {
            "name": "web_search",
            "arguments": {
                "search_str": "Dr Yunus",
                "result": 5,
                "paginate": False,
                "kwargs": {"engine": "google", "grab_index": [1, 3, None]},
            },
        }
    ]
)
print(json_call)
python_call = 'web_search(search_str="Dr Yunus",)'  # paginate=False, result=5, kwargs={\"search_engine\": \"google\"}, grab=[1, 3, \"None\"])"
print(json_tooldef_to_python(tools))
print(json_toolcall_to_python(json_call))

In [ ]:
[{'name': 'format_proj4_string', 'description': 'Formats a string using a PROJ4-style string template.', 'parameters': {'type': 'object', 'properties': {'values': {'type': 'array', 'items': {'type': 'integer'}, 'description': 'A list of 12 integer values representing the zone, projection type, datum type, and projection parameters.'}}, 'required': ['values']}}, {'name': 'format_greeting', 'description': 'Formats a greeting message for a user.', 'parameters': {'type': 'object', 'properties': {'user_info': {'type': 'object', 'additionalProperties': {'type': 'string'}, 'description': "A dictionary containing user information with keys 'user_name', 'user_city', and 'user_country'."}}, 'required': ['user_info']}}, {'name': 'get_average', 'description': 'Returns the average of a list of numbers.', 'parameters': {'type': 'object', 'properties': {'nums': {'type': 'array', 'items': {'type': 'integer'}, 'description': 'A list of integers.'}}, 'required': ['nums']}}, {'name': 'divisible_by_2_and_3', 'description': 'Returns a list of numbers between 1 and n (inclusive) that are divisible by both 2 and 3.', 'parameters': {'type': 'object', 'properties': {'n': {'type': 'integer', 'description': 'A positive integer representing the upper limit of the range.'}}, 'required': ['n']}}, {'name': 'check_interval', 'description': 'Checks if a number is within a specified interval.', 'parameters': {'type': 'object', 'properties': {'low': {'type': ['integer', 'number'], 'description': 'The lower bound of the interval.'}, 'high': {'type': ['integer', 'number'], 'description': 'The upper bound of the interval.'}, 'num': {'type': ['integer', 'number'], 'description': 'The number to check.'}}, 'required': ['low', 'high', 'num']}}, {'name': 'convert_point_cloud', 'description': 'Converts a point cloud from a list of 3D points to a list of lists of coordinates.', 'parameters': {'type': 'object', 'properties': {'point_cloud': {'type': 'array', 'items': {'type': 'array', 'items': {'type': 'number'}}, 'description': 'A list of 3D points, where each point is represented as a list [x, y, z].'}}, 'required': ['point_cloud']}}]

In [ ]:
[{'name': 'zipcodesbyids', 'description': 'Fetches boundaries of given ZIP Codes in GeoJSON format.', 'parameters': {'ids': {'description': "Comma-separated list of ZIP Code IDs. Maximum size is 200. Example: '10021,10022,10023'.", 'type': 'str', 'default': ''}, 'properties': {'description': "Comma-separated list of properties to include in the response. Default values are 'zip,centroid,aland,awater'.", 'type': 'str, optional', 'default': 'zip,centroid,aland,awater'}}}, {'name': 'state', 'description': "Fetch a list of sub-regions/states/provinces/departments of the world's countries based on specified filters.", 'parameters': {'limit': {'description': 'Maximum number of records to return. Default is 250.', 'type': 'int, optional', 'default': '250'}, 'iso_a2': {'description': "Two-letter country code to filter results. Default is 'us'.", 'type': 'str, optional', 'default': 'us'}, 'iso_3166_2': {'description': "Subregion's ISO-3166-2 letter code to filter results.", 'type': 'str, optional', 'default': ''}, 'fields': {'description': "Comma-separated list of fields to include in the result. Default is 'iso_a2'.", 'type': 'str, optional', 'default': 'iso_a2'}, 'name': {'description': "Partial name filter for states in the specified language. Default is 'tex'.", 'type': 'str, optional', 'default': 'tex'}, 'lang': {'description': "ISO 639-1 language code for language selection. Overrides Accept-Language header. Default is 'en'.", 'type': 'str, optional', 'default': 'en'}}}, {'name': 'search_php', 'description': 'Search for geocoding information using the specified Geokeo Forward Geocoding API.', 'parameters': {'api': {'description': 'The API key for accessing the Geokeo Forward Geocoding service.', 'type': 'str', 'default': 'api key from geokeo'}, 'q': {'description': 'The address or location query string to be geocoded.', 'type': 'str', 'default': 'empire state building'}}}, {'name': 'geocode', 'description': 'Fetches geographic information for a given address in Tunisia.', 'parameters': {'address': {'description': 'The address of the location to look up.', 'type': 'str', 'default': 'Tunis'}}}, {'name': 'get_ip_geolocation', 'description': "Fetches the geolocation information for a given IP address using the Toolbench RapidAPI service. If no IP address is specified, it returns the geolocation information for the client's IP address.", 'parameters': {'ip': {'description': "The IP address to get geolocation information for. Defaults to '206.71.50.230'.", 'type': 'str', 'default': '206.71.50.230'}}}, {'name': 'measure_distance', 'description': 'Calculates the distance between two geographic locations based on their latitude and longitude coordinates. The unit of measurement for the distance can be specified.', 'parameters': {'lon2': {'description': 'Longitude of the second location.', 'type': 'int', 'default': '31.23788289124186'}, 'lat1': {'description': 'Latitude of the first location.', 'type': 'int', 'default': '31.1991806'}, 'lon1': {'description': 'Longitude of the first location.', 'type': 'int', 'default': '29.8951716'}, 'lat2': {'description': 'Latitude of the second location.', 'type': 'int', 'default': '30.02313795'}, 'unit': {'description': "Unit of distance measurement. Valid options are 'km' (default), 'mi', 'ft', and 'yd'.", 'type': 'str, optional', 'default': 'km'}}}]

In [ ]:
import json

interact_agent = []
with open('datasets/nemotron/interactive_agent.jsonl', 'r') as f:
    for l in f:
        interact_agent.append(json.loads(l))
        if len(interact_agent) > 100:
            break
print(len(interact_agent))

In [ ]:
interact_agent[0]

In [ ]:
tc_agent = []
with open('datasets/nemotron/tool_calling.jsonl', 'r') as f:
    for l in f:
        tc_agent.append(json.loads(l))
        if len(tc_agent) > 100:
            break
print(len(tc_agent))

In [ ]:
tc_agent[10]

In [ ]:
def mobileactions(tokenizer, prompt_token_len):
    apply_chat_template = lambda seq: tokenizer.apply_chat_template(
            seq,
            add_generation_prompt=False,
            tokenize=False,
            continue_final_message=False,
        )
    
    def mapper(data):
        tools = data["tools"] #json.loads(data["tools"])
        tools = [t.get('function', t) for t in tools]
        if not isinstance(tools, list):
            tools = [tools]

        messages = data['messages'] #json.loads(data['messages'])
        date_time = ' '.join(messages[0]['content'].split('\n')[:2])
        user_question = messages[1]['content']

        tool_calls = messages[2]['tool_calls']
        tool_calls = [t.get('function', t) for t in tool_calls]
        called_tools_name = [t['name'] for t in tool_calls]
        tot_tools = len(called_tools_name)

        random.shuffle(tools)
        del_idx = []
        for i, t in enumerate(tools):
            if t['name'] in called_tools_name:
                continue
            if tot_tools + len()
        
        del_idx = sorted(del_idx, reverse=True)
        for di in del_idx:
            del tools[di]

        seq = [
            {
                "role": "system",
                "content": TOOL_TEMPLATE.format(tools=tool_shuffle(tools)) + '\n' + str(date_time)
            },
            {"role": "user", "content": user_question},
            # {"role": "assistant", "content": "<tool_call>"}
        ]
        
        return {
            "prompt": apply_chat_template(seq),
            "messages": seq,
            "def_tools": tools,
            "ground_tool_call": json.dumps(tool_calls, default=str),
            "num_input_tools": len(tools),
            "scorer": partial(scorer, tools_ground=tool_calls, def_tools=tools)
        }

    train_ds = load_dataset("google/mobile-actions")["train"]
    train_ds = list(map(mapper, train_ds))
    train_ds = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, train_ds))
    return train_ds

In [ ]:
from datasets import load_dataset
import sys
sys.path.append("../")

from utils.tokenizer import TOOL_TEMPLATE, get_tokenizer

tokenizer = get_tokenizer("HuggingFaceTB/SmolLM2-135M-Instruct", add_bos=False)

def filter_by_resp_len(ds, resp_lim:int):
    def filter_fn(data):
        for turn in data['messages']:
            if turn['role'] == 'assistant':
                if len(turn['content']) > resp_lim:
                    return False
                return True
    ds = ds.filter(filter_fn)
    return ds

def nemotron_instruct(tokenizer):
    def mapper(data):
        if len(data['tools']) > 0:
            data['messages'] = TOOL_TEMPLATE.format(tools=json.dumps(data['tools'], indent=random.randint(0, 4))) + data['messages']
        return {
            "messages": data['messages'],
            "source": "nvidia/Nemotron-Instruction-Following-Chat-v1"
        }

    ds = load_dataset("nvidia/Nemotron-Instruction-Following-Chat-v1")
    ds = ds.remove_columns(["uuid", "license", "used_in", "reasoning", "capability_target"])
    ds = ds.map(
        mapper
    )
    return ds

ds = nemotron_instruct(tokenizer)
ds = filter_by_resp_len(ds, 1024*4)

/Users/ohi/Documents/GitHub/NanoAgent/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'utils.tokenizer'; 'utils' is not a package

In [ ]:
ds['chat_if'][1110]